<a href="https://colab.research.google.com/github/Sumit2003-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit2003-dev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Task type:** This is primarily a **scoring/ranking problem**, built on top of a
binary classification model. I'm not just predicting a single class — I'm producing
a probability/score per page, then ranking all pages by that score so a reviewer can
work top-down. Underneath, the core model is a binary classifier (declining vs. not),
but the actual deliverable — the ranked review queue — is what makes this a ranking task.

In [ ]:
print("Task type: Ranking (built on a binary classification model)")

Task type: Ranking (built on a binary classification model)


**Target (starter dataset, proxy):** `is_declining_label` = `trend_direction == "down"`.

This is a proxy, not a true future outcome — it's calculated from the current window,
not from what happens next. A stronger version (planned for later weeks, once I move
to the warehouse release): prior 90 days of features → next 30 days decline. For now,
in the starter slice, I'm using the proxy label to build and test the pipeline honestly,
while flagging its weakness.


In [ ]:
import pandas as pd, numpy as np, os

# same repo-clone setup as w01, in case this is a fresh Colab session
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sumit2003-dev/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Target column: is_declining_label")
print("Positive rate:", round(df["is_declining_label"].mean(), 3))

Target column: is_declining_label
Positive rate: 0.542


**Success metric:** Precision@K (specifically Precision@50), not accuracy.

Why: the actual use case is a reviewer working through a fixed-size queue (e.g. top 50
pages), not a system that judges every page in the dataset. Accuracy would be misleading
here anyway since the declining rate is already ~54% (near-balanced), so a trivial
"always predict declining" rule would still score ~54% accuracy without being useful.
Precision@K directly measures: of the top K pages handed to a reviewer, how many are
actually worth their time.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Chosen metric: Precision@50")
print("Reason: matches a reviewer working a fixed-size queue, not full-dataset accuracy")

Chosen metric: Precision@50
Reason: matches a reviewer working a fixed-size queue, not full-dataset accuracy


One row = one content page, identified by `content_id`, with its trailing-window
signals (impressions, staleness, position, CTR) and its current trend label.

In [ ]:
print("Shape:", df.shape)
print("Unique content_id count:", df["content_id"].nunique())

df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "trend_direction", "is_declining_label"]].head(5)

Shape: (30000, 45)
Unique content_id count: 30000


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down,1


A fixed rule (e.g. "stale AND visible") can only combine 2-3 signals with hard
thresholds — it's a blunt filter, not a ranking. My Week 1-2 notebooks already showed
this in practice: a hand rule scored Precision@50 = 0.680, while a depth-2 decision
tree got 0.600 — but per the internship guide's full-pipeline results, a random forest
reaches Precision@50 = 0.740 vs. the baseline rule's 0.240 at scale.

ML wins here because:
- It can weigh many weak signals together (impressions, staleness, position, CTR,
  word count, engagement) instead of a hand-picked 2-3.
- It can find non-obvious interactions (e.g. high impressions + newer content behaving
  differently than high impressions + old content) that a simple AND/OR rule can't express.
- It produces a continuous score, which ranking needs — a fixed rule only gives a
  yes/no filter, not an ordering within the "yes" group.

Where a fixed rule stays valuable: as a transparent baseline to beat, and as a
sanity-check / reason-code layer on top of the model's output.

In [ ]:
print("Hand rule Precision@50 (Week 2 notebook): 0.680")
print("Depth-2 tree Precision@50 (Week 2 notebook): 0.600")
print("Random forest Precision@50 (guide's full pipeline): 0.740 vs baseline 0.240")

Hand rule Precision@50 (Week 2 notebook): 0.680
Depth-2 tree Precision@50 (Week 2 notebook): 0.600
Random forest Precision@50 (guide's full pipeline): 0.740 vs baseline 0.240


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.